# 전체 사진 → 6부위 크롭 저장

- 대상: Training + Validation, **스마트패드·스마트폰**, 모든 각도 (965명, 5,790장)
  - 디지털카메라는 제외한다. 검은 배경 + 스튜디오 조명이라 같은 사람도 색이 크게 달라져서, 멜라닌 학습에 방해가 된다.
- 부위 추출은 `face_regions.py` (랜드마크 다각형 ∩ 얼굴 피부)
- 눈가는 눈 가림 막대의 좌/우 끝을 한 변으로 쓴다.
- 저장: `data/01.원천데이터/{사람번호}/{기기}/{각도}_{부위}.png`
  - 예: `data/01.원천데이터/0002/스마트폰/F_l_cheek.png`
  - **RGBA PNG**: RGB = 크롭(원본 해상도), 알파 = 픽셀 마스크 (피부 255, 나머지 0)
  - 가려진 부위(`is_visible`)는 파일을 만들지 않는다.
- 목록: `data/index.csv` — 사진 1장 × 부위 6개 = 6행
  - `status`: `ok` / `hidden_view`(측면 사진에서 가려지는 쪽) / `low_res`(사진이 100만 픽셀 미만)
    / `no_face`(랜드마크 실패) / `empty`(남는 피부 픽셀 없음)
  - 저해상도 사진은 크롭을 저장하지 않는다. 얼굴이 작게 찍혀서 배율을 맞추려면 3~4배 확대해야 하는데,
    없던 모공이 생기는 게 아니라 뿌옇게 늘어날 뿐이라 학습에 잡음이 된다 (전체의 약 1%).
  - `visible`: `ok`면 1 → 모델의 가시성 마스크로 쓴다.
  - `face_h`: 얼굴 세로 길이(랜드마크 10번~152번, 픽셀). 학습할 때 배율을 맞추는 기준이다.
  - `split`: 원본 데이터셋의 Training/Validation 구분 (폴더는 나누지 않고 열로만 남긴다)
- 중간에 멈춰도 다시 실행하면 `index.csv`에 있는 사진은 건너뛰고 이어서 처리한다.

In [14]:
import csv
import time
from collections import Counter
from pathlib import Path

import cv2

import importlib

import face_regions

importlib.reload(face_regions)  # face_regions.py 수정사항을 커널 재시작 없이 반영
from face_regions import (DATA_DIR, REGION_NAMES, create_landmarker, create_segmenter, crop_polygon, detect_landmarks,
                          eye_bar_box, face_height, face_skin_mask, is_visible, region_polygons, save_region_png)

DATA_OUT = Path.cwd().parent / "data"          # 프로젝트 루트의 data 폴더
OUT_DIR = DATA_OUT / "01.원천데이터"            # 부위 크롭 저장 위치
INDEX_PATH = DATA_OUT / "index.csv"
COLUMNS = ["split", "device", "subject", "stem", "view", "region", "status", "visible", "file", "width", "height",
           "skin_px", "face_h"]

MIN_PIXELS = 1_000_000  # 이보다 작은 사진은 저해상도로 보고 건너뛴다 (예: 360x480 = 17만)

# 원본 폴더명 → 저장할 폴더명 (디지털카메라는 제외)
DEVICES = {"2. 스마트패드": "스마트패드", "3. 스마트폰": "스마트폰"}

images = sorted(p for d in DEVICES for p in DATA_DIR.glob(f"*/01.원천데이터/{d}/*/*.jpg"))
print(f"{len(images):,}장, 사람 {len({p.parts[-2] for p in images}):,}명")

5,790장, 사람 965명


In [15]:
def process_image(path, landmarker, segmenter):
    """사진 1장을 처리해 부위 크롭을 저장하고 index.csv 행 6개를 반환한다.

    행은 부위마다 하나씩, 저장하지 못한 부위도 이유(status)를 남긴다.
    나중에 "왜 이 사람 볼 사진이 없지?"를 index.csv만 보고 알 수 있어야 하기 때문이다.
    """
    # 경로에서 정보를 꺼낸다: .../Training/01.원천데이터/3. 스마트폰/0002/0002_03_F.jpg
    split, device, subject = path.parts[-5], path.parts[-3], path.parts[-2]
    stem = path.stem
    view = stem.split("_")[2]  # 0002_03_F → F (사람번호_기기_각도)
    base = dict(split=split, device=DEVICES[device], subject=subject, stem=stem, view=view)

    bgr = cv2.imread(str(path))

    # 저해상도 사진은 건너뛴다. 얼굴 세로를 900px로 맞추려면 3~4배 확대해야 하는데,
    # 없던 모공이 생기는 게 아니라 뿌옇게 늘어날 뿐이라 학습에 잡음이 된다.
    if bgr.shape[0] * bgr.shape[1] < MIN_PIXELS:
        base["width"], base["height"] = bgr.shape[1], bgr.shape[0]
        return [dict(base, region=name, status="low_res", visible=0) for name in REGION_NAMES]

    # 얼굴을 못 찾으면 이 사진은 통째로 쓸 수 없다
    points = detect_landmarks(landmarker, bgr)
    if points is None:
        return [dict(base, region=name, status="no_face", visible=0) for name in REGION_NAMES]

    # 이후 작업은 모두 RGB 기준. mediapipe도 matplotlib도 RGB를 기대한다
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    skin = face_skin_mask(segmenter, rgb, points)  # 배경·머리카락을 걸러낼 얼굴 피부 마스크
    bar = eye_bar_box(rgb, points)  # 눈 가림 막대. 눈가 영역의 한 변으로 쓴다
    base["face_h"] = round(face_height(points), 1)  # 학습 때 배율을 맞추는 기준값

    out_dir = OUT_DIR / subject / DEVICES[device]
    out_dir.mkdir(parents=True, exist_ok=True)

    rows = []
    for name, polygon in region_polygons(points, bar).items():
        # 측면 사진에서 반대쪽 눈가·볼은 가려져 있다. 억지로 자르면 코·입술이 들어온다
        if not is_visible(view, name):
            rows.append(dict(base, region=name, status="hidden_view", visible=0))
            continue

        # 다각형 ∩ 얼굴 피부. 남는 픽셀이 없으면(이미지 밖으로 나갔거나 전부 가려짐) None
        result = crop_polygon(rgb, polygon, skin)
        if result is None:
            rows.append(dict(base, region=name, status="empty", visible=0))
            continue

        # RGBA PNG로 저장: RGB = 원본 해상도 크롭, 알파 = 피부 마스크
        crop, mask = result
        file = out_dir / f"{view}_{name}.png"
        save_region_png(file, crop, mask)
        rows.append(dict(base, region=name, status="ok", visible=1, file=str(file.relative_to(OUT_DIR)),
                         width=crop.shape[1], height=crop.shape[0], skin_px=int(mask.sum())))
    return rows

## 실행

`LIMIT`에 숫자를 넣으면 그 장수만 처리한다 (테스트용). 전체는 `None`.

In [16]:
LIMIT = None

OUT_DIR.mkdir(parents=True, exist_ok=True)
done = set()
if INDEX_PATH.exists():
    with open(INDEX_PATH, newline="") as f:
        done = {row["stem"] + row["device"] for row in csv.DictReader(f)}
todo = [p for p in images if p.stem + DEVICES[p.parts[-3]] not in done][:LIMIT]
print(f"완료 {len(done):,}장, 남은 {len(todo):,}장")

new_file = not INDEX_PATH.exists()
start = time.time()
with open(INDEX_PATH, "a", newline="") as f, create_landmarker() as landmarker, create_segmenter() as segmenter:
    writer = csv.DictWriter(f, fieldnames=COLUMNS)
    if new_file:
        writer.writeheader()
    for i, path in enumerate(todo, 1):
        writer.writerows(process_image(path, landmarker, segmenter))
        if i % 100 == 0 or i == len(todo):
            f.flush()
            elapsed = time.time() - start
            print(f"{i:,}/{len(todo):,}  {elapsed / i:.2f}초/장, 남은 시간 약 {elapsed / i * (len(todo) - i) / 60:.0f}분")

완료 0장, 남은 5,790장


W0000 00:00:1790066454.415151 1254002 face_landmarker_graph.cc:180] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
I0000 00:00:1790066454.423569 1254002 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M5
W0000 00:00:1790066454.425515 1254006 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1790066454.435023 1254006 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
I0000 00:00:1790066454.444970 1254016 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M5
W0000 00:00:1790066454.474716 1254018 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


100/5,790  0.19초/장, 남은 시간 약 18분
200/5,790  0.19초/장, 남은 시간 약 18분
300/5,790  0.19초/장, 남은 시간 약 17분
400/5,790  0.19초/장, 남은 시간 약 17분
500/5,790  0.20초/장, 남은 시간 약 17분
600/5,790  0.20초/장, 남은 시간 약 17분


E0000 00:00:1790066574.808354 1254017 portable_clearcut_uploader.cc:90] Failed to send to clearcut: FAILED_PRECONDITION: Not valid for uploading until: 2026-09-22T17:56:54.797967+09:00
=== Source Location Trace: === 
wireless/android/play/playlog/cplusplus/portable_clearcut_uploader.cc:180
E0000 00:00:1790066574.837444 1254003 portable_clearcut_uploader.cc:90] Failed to send to clearcut: FAILED_PRECONDITION: Not valid for uploading until: 2026-09-22T17:56:54.832888+09:00
=== Source Location Trace: === 
wireless/android/play/playlog/cplusplus/portable_clearcut_uploader.cc:180


700/5,790  0.20초/장, 남은 시간 약 17분
800/5,790  0.20초/장, 남은 시간 약 17분


...

E0000 00:00:1790067475.210479 1254017 portable_clearcut_uploader.cc:90] Failed to send to clearcut: FAILED_PRECONDITION: Not valid for uploading until: 2026-09-22T18:11:55.161038+09:00
=== Source Location Trace: === 
wireless/android/play/playlog/cplusplus/portable_clearcut_uploader.cc:180
E0000 00:00:1790067475.336714 1254003 portable_clearcut_uploader.cc:90] Failed to send to clearcut: FAILED_PRECONDITION: Not valid for uploading until: 2026-09-22T18:11:55.282819+09:00
=== Source Location Trace: === 
wireless/android/play/playlog/cplusplus/portable_clearcut_uploader.cc:180


5,400/5,790  0.19초/장, 남은 시간 약 1분
5,500/5,790  0.19초/장, 남은 시간 약 1분
5,600/5,790  0.19초/장, 남은 시간 약 1분
5,700/5,790  0.19초/장, 남은 시간 약 0분


E0000 00:00:1790067535.218419 1254017 portable_clearcut_uploader.cc:90] Failed to send to clearcut: FAILED_PRECONDITION: Not valid for uploading until: 2026-09-22T18:11:55.161038+09:00
=== Source Location Trace: === 
wireless/android/play/playlog/cplusplus/portable_clearcut_uploader.cc:180
E0000 00:00:1790067535.340076 1254003 portable_clearcut_uploader.cc:90] Failed to send to clearcut: FAILED_PRECONDITION: Not valid for uploading until: 2026-09-22T18:11:55.282819+09:00
=== Source Location Trace: === 
wireless/android/play/playlog/cplusplus/portable_clearcut_uploader.cc:180


5,790/5,790  0.19초/장, 남은 시간 약 0분


E0000 00:00:1790067547.651107 1254017 portable_clearcut_uploader.cc:90] Failed to send to clearcut: FAILED_PRECONDITION: Not valid for uploading until: 2026-09-22T18:11:55.161038+09:00
=== Source Location Trace: === 
wireless/android/play/playlog/cplusplus/portable_clearcut_uploader.cc:180
E0000 00:00:1790067547.653439 1254003 portable_clearcut_uploader.cc:90] Failed to send to clearcut: FAILED_PRECONDITION: Not valid for uploading until: 2026-09-22T18:11:55.282819+09:00
=== Source Location Trace: === 
wireless/android/play/playlog/cplusplus/portable_clearcut_uploader.cc:180


## 결과 요약

In [18]:
with open(INDEX_PATH, newline="") as f:
    rows = list(csv.DictReader(f))

print(f"사진 {len({(r['stem'], r['device']) for r in rows}):,}장, 부위별 사진 {len(rows):,}개")
print("status:", dict(Counter(r["status"] for r in rows)))
print("얼굴 못 찾은 사진:", sorted({r["stem"] + " (" + r["device"] + ")" for r in rows if r["status"] == "no_face"}))
print("피부가 안 남은 부위:", [(r["stem"], r["region"]) for r in rows if r["status"] == "empty"])

사진 5,790장, 부위별 사진 34,740개
status: {'ok': 26754, 'hidden_view': 7644, 'low_res': 342}
얼굴 못 찾은 사진: []
피부가 안 남은 부위: []
